In [1]:
!pip install datasets pandas tqdm

In [2]:
from datasets import load_dataset

dataset = load_dataset("squad_v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})
{'id': '56be85543aeaaa14008c9063', 'title': 'Beyoncé', 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "C

In [4]:
data = []

for item in dataset['train']:
    context = item['context']
    question = item['question']

    # handle unanswerable
    answer = item['answers']['text']
    answer = answer[0] if len(answer) > 0 else ""

    data.append({
        "context": context,
        "question": question,
        "answer": answer
    })

print(len(data))
print(data[0])

130319
{'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".', 'question': 'When did Beyonce start becoming popular?', 'answer': 'in the late 1990s'}


In [5]:
data = data[:2000]   # for speed i reduced the size

In [6]:
import json

with open("processed_squad.json", "w") as f:
    json.dump(data, f)

In [7]:
import json

with open("processed_squad.json", "r") as f:
    data = json.load(f)

print(len(data))

2000


In [8]:
def split_into_paragraphs(text):
    return text.split("\n")

In [9]:
#sliding window concept for chunking
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        start += (chunk_size - overlap)

    return chunks

In [10]:
chunked_data = []

for item in data:
    context = item["context"]

    # split into paragraphs
    paragraphs = split_into_paragraphs(context)

    for para in paragraphs:
        chunks = chunk_text(para)

        for chunk in chunks:
            chunked_data.append({
                "chunk": chunk,
                "question": item["question"],
                "answer": item["answer"]
            })

print("Total chunks:", len(chunked_data))
print(chunked_data[0])

Total chunks: 4900
{'chunk': "Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world's best-selling girl groups of all time. Their hiatus saw the release of Beyoncé's debut al", 'question': 'When did Beyonce start becoming popular?', 'answer': 'in the late 1990s'}


In [11]:
chunked_data = [c for c in chunked_data if len(c["chunk"]) > 50]

In [12]:
with open("chunked_data.json", "w") as f:
    json.dump(chunked_data, f)

In [13]:
!pip install sentence-transformers

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [item["chunk"] for item in chunked_data]

embeddings = model.encode(texts, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/147 [00:00<?, ?it/s]

In [15]:
print(len(embeddings))
print(len(chunked_data))

4697
4697


In [16]:
import numpy as np
import json

np.save("embeddings.npy", embeddings)

with open("metadata.json", "w") as f:
    json.dump(chunked_data, f)

In [17]:
import numpy as np

embeddings = np.load("embeddings.npy", allow_pickle=True)

print("Number of embeddings:", len(embeddings))
print("Dimension of each embedding:", len(embeddings[0]))

Number of embeddings: 4697
Dimension of each embedding: 384


In [18]:
import json

with open("metadata.json", "r") as f:
    metadata = json.load(f)

print("Metadata records:", len(metadata))

Metadata records: 4697


In [19]:
!pip install chromadb

In [20]:
import chromadb

client = chromadb.Client()

# delete old collection if exists
try:
    client.delete_collection("rag_collection")
except:
    pass

# create fresh collection
collection = client.create_collection(name="rag_collection")

In [21]:
batch_size = 100

for i in range(0, len(embeddings), batch_size):
    batch_embeddings = embeddings[i:i+batch_size]
    batch_metadata = metadata[i:i+batch_size]

    collection.add(
        embeddings=[e.tolist() for e in batch_embeddings],
        documents=[m["chunk"] for m in batch_metadata],
        metadatas=[{"source": "squad"} for _ in batch_metadata],
        ids=[str(j) for j in range(i, i + len(batch_embeddings))]
    )

print("✅ All embeddings stored")

✅ All embeddings stored


In [22]:
def retrieve(query, k=5):
    query_embedding = embed_model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k
    )

    return results

In [24]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
query = "Who is Beyonce?"

results = retrieve(query)

for i in range(len(results["documents"][0])):
    print("\n--- RESULT ---")
    print(results["documents"][0][i])


--- RESULT ---
can, Native American, French, Cajun, and distant Irish and Spanish ancestry). Through her mother, Beyoncé is a descendant of Acadian leader Joseph Broussard. She was raised in a Methodist household.

--- RESULT ---
can, Native American, French, Cajun, and distant Irish and Spanish ancestry). Through her mother, Beyoncé is a descendant of Acadian leader Joseph Broussard. She was raised in a Methodist household.

--- RESULT ---
can, Native American, French, Cajun, and distant Irish and Spanish ancestry). Through her mother, Beyoncé is a descendant of Acadian leader Joseph Broussard. She was raised in a Methodist household.

--- RESULT ---
can, Native American, French, Cajun, and distant Irish and Spanish ancestry). Through her mother, Beyoncé is a descendant of Acadian leader Joseph Broussard. She was raised in a Methodist household.

--- RESULT ---
can, Native American, French, Cajun, and distant Irish and Spanish ancestry). Through her mother, Beyoncé is a descendant of

In [26]:
!pip install -U google-generativeai

In [43]:
def build_prompt(query, contexts):
    context_text = "\n\n".join(contexts)

    prompt = f"""
You are a precise question-answering system.

Answer ONLY using the most relevant information from the context.

If multiple facts exist, choose the one that directly answers the question.

Context:
{context_text}

Question: {query}

Answer in one clear sentence:
"""
    return prompt

In [37]:
!pip install transformers sentencepiece

In [40]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [75]:
def answer_query(query, k=3):
    results = retrieve(query, k)
    contexts = results["documents"][0][:2]   # only top 2

    context_text = "\n\n".join(contexts)

    prompt = f"""
Answer the question using the context below.

Context:
{context_text}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=100,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [76]:
print(answer_query("Who is Beyonce?"))

a descendant of Acadian leader Joseph Broussard
